In [1]:
import json
import folium
from shapely.geometry import shape

# Load exposure data
with open('Alameda_Exposure_business_sim0_final.json', 'r') as f:
    data = json.load(f)

buildings = data['Buildings']['Building']

# Color mapping by occupancy category
def get_color(occupancy):
    if 'RES' in occupancy:
        return 'green'
    elif 'COM' in occupancy:
        return 'orange'
    elif 'IND' in occupancy:
        return 'royalblue'
    else:
        return 'gray'

# Parse footprints
features = []
lats, lons = [], []
for bid, bdata in buildings.items():
    info = bdata['GeneralInformation']
    occupancy = info.get('OccupancyClass', 'Unknown')
    footprint_str = info.get('Footprint')
    if footprint_str:
        feature = json.loads(footprint_str)
        feature['properties']['building_id'] = bid
        feature['properties']['occupancy'] = occupancy
        features.append(feature)
        centroid = shape(feature['geometry']).centroid
        lats.append(centroid.y)
        lons.append(centroid.x)

# Create interactive map
m = folium.Map(location=[sum(lats)/len(lats), sum(lons)/len(lons)], zoom_start=14, tiles='OpenStreetMap')

folium.GeoJson(
    {'type': 'FeatureCollection', 'features': features},
    style_function=lambda x: {
        'fillColor': get_color(x['properties']['occupancy']),
        'color': 'black',
        'weight': 1,
        'fillOpacity': 0.6
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['building_id', 'occupancy'],
        aliases=['Building ID:', 'Occupancy Class:']
    )
).add_to(m)

# Add legend
legend_html = """
<div style="position:fixed; bottom:30px; left:30px; z-index:1000; background:white;
     padding:10px; border:2px solid grey; border-radius:5px; font-size:13px;">
  <b>Occupancy Class</b><br>
  <i style="background:green; width:12px; height:12px; display:inline-block;"></i> RES<br>
  <i style="background:orange; width:12px; height:12px; display:inline-block;"></i> COM<br>
  <i style="background:royalblue; width:12px; height:12px; display:inline-block;"></i> IND<br>
  <i style="background:gray; width:12px; height:12px; display:inline-block;"></i> Other
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m

In [2]:
import glob
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import unary_union

# Load all TAZ geojsons (excluding outside_island) and union them to get island boundary
taz_files = glob.glob('Alameda TAZ/Alameda_TravelAnalysisZone_*.geojson')
taz_gdfs = [gpd.read_file(f) for f in taz_files]
taz_all = gpd.GeoDataFrame(pd.concat(taz_gdfs, ignore_index=True))
alameda_boundary = unary_union(taz_all.geometry)

# Build GeoDataFrame of all buildings with their attributes
records = []
for bid, bdata in buildings.items():
    info = bdata['GeneralInformation']
    records.append({
        'building_id': bid,
        'occupancy': info.get('OccupancyClass', ''),
        'num_stories': info.get('NumberOfStories', 0),
        'plan_area': info.get('PlanArea', 0),
        'population': info.get('Population', 0),
        'geometry': Point(info['Longitude'], info['Latitude'])
    })

gdf_all = gpd.GeoDataFrame(records, crs='EPSG:4326')

# Filter to buildings on Alameda Island
gdf_alameda = gdf_all[gdf_all.geometry.within(alameda_boundary)].copy()

# Filter to residential buildings (OccupancyClass contains "RES")
gdf_res = gdf_alameda[gdf_alameda['occupancy'].str.contains('RES', na=False)].copy()

# Compute gross area per building: NumberOfStories * PlanArea
gdf_res['gross_area'] = gdf_res['num_stories'] * gdf_res['plan_area']

total_gross_area = gdf_res['gross_area'].sum()

# Count residential buildings with zero population
zero_pop = (gdf_res['population'] == 0).sum()
total_pop = gdf_res['population'].sum()

print(f"Buildings in exposure file: {len(gdf_all)}")
print(f"Buildings on Alameda Island: {len(gdf_alameda)}")
print(f"Residential buildings on Alameda Island: {len(gdf_res)}")
print(f"Residential buildings with zero population: {zero_pop}")
print(f"Total residential population: {total_pop:,.0f}")
print(f"Total residential gross area: {total_gross_area:,.2f} sq inches")
print(f"Total residential gross area: {total_gross_area / 144:,.2f} sq ft")

Buildings in exposure file: 14717
Buildings on Alameda Island: 13943
Residential buildings on Alameda Island: 13310
Residential buildings with zero population: 180
Total residential population: 72,026
Total residential gross area: 598,240,997.99 sq inches
Total residential gross area: 4,154,451.37 sq ft
